# Multiseed RANDOM-STREAM generalisation test — DDPG vs TD3, Approach 1 & 2

Evaluates the four **best online actor-critic policies** of this folder
(`ddpg/td3_online_ac_approach1/2_best.pt`) on **30 random 65/80 °C setpoint streams**
(one per seed), like the RANDOM-STREAM test of the anti-windup notebooks:
random-length segments of a cloudy (65 °C) day and sunny (80 °C) days are
concatenated in random order, and the per-step setpoint switches accordingly.

* **Tuned policies (apple-to-apple)**: all four policies are trained in the same
  widened, stability-safe gain box (`GAIN_LOW=[-30,-4.0,-0.4]`, Kw·Ki·TS ≤ 1.6 < 2)
  with undershoot-aware checkpoint selection — see the four approach notebooks.
* **Data sources, randomized per segment**: 65 °C segments come from the ORIGINAL cloudy
  day (`20_10_2025 Cloudy`); each 80 °C segment picks a random day from **Juan's 4 new
  June days** + the original sunny day (`21_10_2025 Sunny`).
* **Pure zero-shot test** — deterministic rollout, no exploration, no online adaptation.
* Outputs, per seed: the **state chart** (Tout vs the random T_ref, incl. Expert PI) and
  the **action chart** (applied flow q), saved under `charts/randstream_multiseed/`.
* Final **summary table**: MAE, RMSE and peak overshoot (mean ± std over the 30 seeds)
  per policy, plus the per-seed CSV `charts/randstream_multiseed/randstream_multiseed_results.csv`.

Run top-to-bottom (~30-45 min on CPU: 5 rollouts x ~72k steps x 30 seeds).


In [ ]:
# ── Setup: shared library (../main_script) + this folder's config ────────────
import os, sys, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
sys.path.insert(0, os.getcwd())                                            # for: import config
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))  # reach ../main_script
from main_script import *
import config as cfg
from config import *
configure(cfg)

# ── BOX switch: pick the gain box to evaluate in this run ────────────────────
# Edit BOX below and re-run the notebook top-to-bottom. Report labels (exact
# values from the results table in 04_results_discussion.tex); the 4 policies
# loaded below are the CURATED checkpoints in main/policies/, one set per box.
#   Narrow = original box, same range the Phase-2 CQL critic was trained in
#   Mid    = mid box (report Table: [-6.0,-0.100,-0.60])
#   Wide   = wide box (report Table: [-15.0,-1.5625,-0.525])
BOX = 'Narrow'  # 'Narrow' | 'Mid' | 'Wide'
BOX_GAINS = {
    'Narrow': (np.array([-3.5,  -0.060,  -0.35  ], dtype=np.float32), np.array([-0.05, -0.0001, 0.10], dtype=np.float32)),
    'Mid':    (np.array([-6.0,  -0.100,  -0.60  ], dtype=np.float32), np.array([-0.05, -0.0001, 0.10], dtype=np.float32)),
    'Wide':   (np.array([-15.0, -1.5625, -0.525 ], dtype=np.float32), np.array([-0.05, -0.0001, 0.10], dtype=np.float32)),
}
cfg.GAIN_LOW, cfg.GAIN_HIGH = BOX_GAINS[BOX]

# The four BEST online actor-critic policies for this box (curated in main/policies/)
MAIN_POL = os.path.join(BASE_DIR, 'main', 'policies')
POLICIES = {
    'DDPG-A1': f'Best_DDPG_1_{BOX}BoxPolicy.pt',   # DDPG, search -> freeze -> critic-exploit
    'DDPG-A2': f'Best_DDPG_2_{BOX}BoxPolicy.pt',   # DDPG, critic-driven
    'TD3-A1':  f'Best_TD3_1_{BOX}BoxPolicy.pt',    # TD3,  search -> freeze -> critic-exploit
    'TD3-A2':  f'Best_TD3_2_{BOX}BoxPolicy.pt',    # TD3,  critic-driven
}
# DDPG family = warm (dark/light), TD3 family = cool (dark/light)
COLORS = {'DDPG-A1': '#d62728', 'DDPG-A2': '#ff7f0e', 'TD3-A1': '#1f77b4', 'TD3-A2': '#9467bd'}

actors = {tag: load_actor_raw(os.path.join(MAIN_POL, f)) for tag, f in POLICIES.items()}
print(f"loaded {len(actors)} policies | BOX={BOX} | device = {DEVICE}")
print(f"gain box: LOW={list(cfg.GAIN_LOW)} HIGH={list(cfg.GAIN_HIGH)}")


In [ ]:
# ── RANDOM-STREAM builder (anti-windup V1/V2 construction + Juan's new data) ──
# Concatenate random-length segments in random order; the per-step setpoint switches
# accordingly. 65 C segments come from the ORIGINAL cloudy day; each 80 C segment picks
# a RANDOM day out of Juan's 4 new June days + the original sunny day.
N_SEGMENTS       = 16
SEG_MIN, SEG_MAX = 3500, 5500
SEEDS = [42, 36, 27, 21, 14, 23, 39, 94, 15, 88, 26, 83, 51, 18, 68,
         86, 33, 47, 99, 57, 52, 71, 78, 89, 49, 84, 13, 30, 81, 62]   # == BC vs CQL Comparison/multiseed

ORIG_DATA = os.path.abspath(os.path.join(os.getcwd(), os.pardir, 'CQL Offline Actor', 'data'))
d_cloudy  = load_dataset(os.path.join(ORIG_DATA, '20_10_2025__Cloudy_Closed_Loop.xlsx'))
sunny_pool = ([load_dataset(f) for f in JUAN_FILES] +                       # Juan's 4 new days
              [load_dataset(os.path.join(ORIG_DATA, '21_10_2025__Sunny_Closed_Loop.xlsx'))])

def build_random_stream(seed, n_segments=N_SEGMENTS, seg_min=SEG_MIN, seg_max=SEG_MAX):
    rng = np.random.default_rng(seed)
    keys = ['T_sc', 'Tin', 'Ta', 'I_sol', 'theta', 'q']
    cat = {k: [] for k in keys}; tref = []; order = []
    for _ in range(n_segments):
        cloudy = bool(rng.random() < 0.5)
        src = d_cloudy if cloudy else sunny_pool[int(rng.integers(len(sunny_pool)))]
        trv = TREF_CLOUDY if cloudy else TREF_SUNNY
        L  = min(int(rng.integers(seg_min, seg_max + 1)), src['N'])
        st = int(rng.integers(0, max(1, src['N'] - L)))
        for k in keys:
            cat[k].append(src[k][st:st + L])
        tref.append(np.full(L, trv))
        order.append(('cloudy65' if cloudy else f"sunny80:{os.path.basename(src['name'])[:11]}", L))
    data = {k: np.concatenate(cat[k]) for k in keys}
    data['N'] = len(data['T_sc']); data['name'] = f'RANDOM_stream_seed{seed}'
    data['tref_seq'] = np.concatenate(tref)   # per-step setpoint (env + expert both use it)
    return data, order

print(f"{len(SEEDS)} seeds | {N_SEGMENTS} segments of {SEG_MIN}-{SEG_MAX} steps per stream")
print(f"65 C source: {os.path.basename(d_cloudy['name'])} (N={d_cloudy['N']})")
print("80 C pool  :", ", ".join(os.path.basename(d['name'])[:11] for d in sunny_pool))

In [ ]:
# ── 30-seed evaluation: Expert PI + the 4 policies, deterministic rollouts ────
# Per seed: STATE chart (Tout vs random T_ref) + ACTION chart (applied flow q).
RS_CHART_DIR = os.path.join(CHART_DIR, f'randstream_multiseed_{BOX.lower()}'); os.makedirs(RS_CHART_DIR, exist_ok=True)
records = []; _t0 = time.time()

for si, seed in enumerate(SEEDS, 1):
    d_rand, seg_order = build_random_stream(seed)
    Te, Qe = rollout_expert(d_rand)                     # expert PI on the same stream
    Ts, Qs = {}, {}
    for tag, actor in actors.items():                   # deterministic policy rollouts
        Ts[tag], Qs[tag], _ = rollout_policy(actor, d_rand)
    n = min([len(Te)] + [len(v) for v in Ts.values()])
    tref_r = np.asarray(d_rand['tref_seq'][:n], float); t = np.arange(n) * TS

    met = {'Expert': mae_rmse(Te[:n], tref_r) + (peak_overshoot(Te[:n], tref_r),)}
    for tag in POLICIES:
        met[tag] = mae_rmse(Ts[tag][:n], tref_r) + (peak_overshoot(Ts[tag][:n], tref_r),)
    for name, (m, rm, ov) in met.items():
        records.append(dict(seed=seed, policy=name, mae=m, rmse=rm, overshoot=ov))

    fig, ax = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
    ax[0].plot(t, tref_r, color='green',   ls='--', lw=1.3, label='T_ref (random 65/80 C)')
    ax[0].plot(t, Te[:n], color='#888888', ls='-.', lw=0.9,
               label=f"Expert  MAE={met['Expert'][0]:.3f} ovr={met['Expert'][2]:.2f}")
    for tag in POLICIES:
        ax[0].plot(t, Ts[tag][:n], color=COLORS[tag], lw=1.0,
                   label=f"{tag}  MAE={met[tag][0]:.3f} ovr={met[tag][2]:.2f}")
    ax[0].set_ylim(45, 95)   # keep readable even if a policy diverges (values in legend)
    ax[0].set_ylabel('Tout (C)'); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8, ncol=2)
    ax[0].set_title(f'RANDOM-STREAM generalisation test (seed={seed}) - '
                    f'DDPG/TD3 Approach 1 & 2 vs Expert PI')
    ax[1].plot(t, Qe[:n], color='#888888', ls='-.', lw=0.9, label='q Expert')
    for tag in POLICIES:
        ax[1].plot(t, Qs[tag][:n], color=COLORS[tag], lw=0.8, label=f'q {tag}')
    ax[1].axhline(Q_MIN, color='gray', ls=':'); ax[1].axhline(Q_MAX, color='gray', ls=':')
    ax[1].set_ylabel('Flow q (L/min)'); ax[1].set_xlabel('Time (s)')
    ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(os.path.join(RS_CHART_DIR, f'randstream_seed{seed}.png'), dpi=140, bbox_inches='tight')
    plt.show(); plt.close(fig)

    print(f"[{si:2d}/{len(SEEDS)}] seed={seed:2d} ({d_rand['N']} steps) | " +
          " | ".join(f"{k} MAE={v[0]:.3f} ovr={v[2]:.1f}" for k, v in met.items()) +
          f" | {(time.time() - _t0) / 60:.1f} min", flush=True)

print(f"\ndone: {len(SEEDS)} streams in {(time.time() - _t0) / 60:.1f} min | charts -> {RS_CHART_DIR}")


In [ ]:
# ── Summary table: MAE / RMSE / peak overshoot over the 30 random streams ────
df = pd.DataFrame(records)
df.to_csv(os.path.join(RS_CHART_DIR, f'randstream_multiseed_{BOX.lower()}_results.csv'), index=False)

ORDER = ['Expert', 'DDPG-A1', 'DDPG-A2', 'TD3-A1', 'TD3-A2']
per_seed_mae = df.pivot(index='seed', columns='policy', values='mae').loc[SEEDS, ORDER]
print('Per-seed MAE (C):')
display(per_seed_mae.round(3))

summary = df.groupby('policy').agg(MAE_mean=('mae', 'mean'),   MAE_std=('mae', 'std'),
                                   RMSE_mean=('rmse', 'mean'), RMSE_std=('rmse', 'std'),
                                   OVR_mean=('overshoot', 'mean'), OVR_std=('overshoot', 'std')).loc[ORDER]
display(summary.round(3))

print(f'\nMAE / RMSE / overshoot over {len(SEEDS)} random streams (mean +/- std):')
for tag in ORDER:
    r = summary.loc[tag]
    print(f'  {tag:8s} MAE = {r.MAE_mean:.3f} +/- {r.MAE_std:.3f}   '
          f'RMSE = {r.RMSE_mean:.3f} +/- {r.RMSE_std:.3f}   '
          f'ovr = {r.OVR_mean:.2f} +/- {r.OVR_std:.2f}')
best = summary.drop('Expert').MAE_mean.idxmin()
print(f'\nBest policy by mean MAE: {best} '
      f'({summary.loc[best].MAE_mean:.3f} vs Expert {summary.loc["Expert"].MAE_mean:.3f})')


In [ ]:
# ── Summary chart: mean +/- std over seeds (dots = individual seeds) ─────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, lab in ((axes[0], 'mae', 'MAE (C)'), (axes[1], 'rmse', 'RMSE (C)'),
                        (axes[2], 'overshoot', 'Peak overshoot (C)')):
    tags = ORDER[1:]
    vals = [df[df.policy == tag][metric].to_numpy() for tag in tags]
    exp_m = df[df.policy == 'Expert'][metric].mean()
    x = np.arange(len(tags))
    ax.bar(x, [v.mean() for v in vals], yerr=[v.std() for v in vals], capsize=6,
           color=[COLORS[tag] for tag in tags], alpha=.75, width=.6)
    for xi, v in zip(x, vals):
        ax.scatter([xi] * len(v), v, color='k', s=12, zorder=3)
    ax.axhline(exp_m, ls='--', c='gray', label=f'Expert PI ({exp_m:.3f})')
    ax.set_xticks(x); ax.set_xticklabels(tags); ax.set_ylabel(lab)
    ax.grid(axis='y', alpha=.3); ax.legend()
fig.suptitle(f'RANDOM-STREAM generalisation over {len(SEEDS)} seeds - {BOX} box - closed-loop tracking error')
fig.tight_layout()
fig.savefig(os.path.join(RS_CHART_DIR, f'randstream_multiseed_{BOX.lower()}_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
